# Teste de Conexão ao Database wtnps-trade.db

Este notebook demonstra como conectar ao banco de dados SQLite `wtnps-trade.db` e utilizar os métodos da classe `AssetsRatesRepository`.

## Estrutura do Notebook:
1. **Configuração**: Imports e conexão ao banco
2. **Métodos do AssetsRatesRepository**:
   - `save_rates_dataframe()` - Salvar dados OHLC
   - `get_all_rates()` - Buscar todos os rates
   - `get_rates_range()` - Buscar por intervalo de datas
   - `get_rates_indicators_range()` - Buscar indicadores por intervalo

## 1. Configuração e Imports

## ⚠️ IMPORTANTE: REINICIAR KERNEL

**ANTES DE EXECUTAR ESTE NOTEBOOK:**
1. Reinicie o kernel do Jupyter: `Kernel → Restart Kernel`
2. Execute as células NA ORDEM (1 → 2 → 3...)

**Por quê?**
- O notebook cria o engine SQLAlchemy com caminho **absoluto** para o banco
- Se já houver um engine criado com caminho relativo, haverá conflito
- Reiniciar o kernel garante ambiente limpo

**Problema que isso resolve:**
- Havia um arquivo `wtnps_trade.db` vazio em `newapp/notebooks/` (90 KB)
- O banco REAL está na raiz do projeto (21 MB com 100k+ registros)
- Caminhos relativos apontavam para o banco errado

In [ ]:
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime, timedelta

# Adiciona o diretório raiz ao path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")
print(f"✅ Python path configurado")

In [ ]:
# SOLUÇÃO: Criar engine diretamente com caminho absoluto
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from newapp.src.database.repository import AssetsRatesRepository

# Caminho absoluto do banco
DB_PATH = project_root / "wtnps_trade.db"

print(f"✅ Database path: {DB_PATH}")
print(f"✅ Database exists: {DB_PATH.exists()}")

# Criar engine DIRETO com caminho absoluto
engine = create_engine(f"sqlite:///{DB_PATH}", connect_args={'check_same_thread': False})

# Criar session factory
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

print(f"✅ Engine e SessionLocal criados com caminho absoluto")

In [ ]:
# Criar sessão do banco de dados
db = SessionLocal()

# Testar conexão usando o engine JÁ CRIADO (não chamar get_engine()!)
with engine.connect() as conn:
    from sqlalchemy import text
    result = conn.execute(text("SELECT sqlite_version()"))
    version = result.fetchone()[0]
    print(f"✅ Conectado ao SQLite {version}")
    
print(f"✅ Sessão do banco criada com sucesso")
print(f"✅ Engine URL: {engine.url}")

### Verificação Rápida dos Dados Disponíveis

In [ ]:
# Verificar rapidamente o que tem no banco
from sqlalchemy import text

with engine.connect() as conn:
    # Listar todas as tabelas
    result = conn.execute(text("""
        SELECT name FROM sqlite_master 
        WHERE type='table' 
        ORDER BY name
    """))
    tables = [row[0] for row in result.fetchall()]
    
    print("📋 TABELAS NO BANCO:")
    print("=" * 60)
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"  • {table:<30} {count:>10} registros")
    
    # Mostrar exemplo de dados da assets_rates
    if 'assets_rates' in tables:
        print("\n🔍 AMOSTRA DA TABELA assets_rates:")
        print("=" * 60)
        result = conn.execute(text("""
            SELECT symbol, timeframe, timeframe_str, timestamp, close
            FROM assets_rates 
            LIMIT 3
        """))
        for row in result:
            print(f"  {row[0]} | TF: {row[1]} ({row[2]}) | {row[3]} | Close: {row[4]}")

### Teste Direto: Confirmar que dados existem para WDO$

In [ ]:
# Teste SQL direto para confirmar dados
with engine.connect() as conn:
    # Contar registros WDO$ com timeframe 5
    result = conn.execute(text("""
        SELECT 
            symbol,
            timeframe,
            timeframe_str,
            COUNT(*) as total,
            MIN(timestamp) as primeira_data,
            MAX(timestamp) as ultima_data
        FROM assets_rates 
        WHERE symbol = 'WDO$' AND timeframe = 5
        GROUP BY symbol, timeframe, timeframe_str
    """))
    
    row = result.fetchone()
    
    if row:
        print("✅ DADOS ENCONTRADOS!")
        print("=" * 70)
        print(f"  Symbol: {row[0]}")
        print(f"  Timeframe: {row[1]} ({row[2]})")
        print(f"  Total de registros: {row[3]}")
        print(f"  Período: {row[4]} até {row[5]}")
        print("=" * 70)
    else:
        print("❌ NENHUM registro encontrado para WDO$ com timeframe=5")
        print("\n🔍 Verificando o que existe:")
        
        result2 = conn.execute(text("""
            SELECT DISTINCT symbol, timeframe, timeframe_str, COUNT(*) as count
            FROM assets_rates 
            GROUP BY symbol, timeframe, timeframe_str
            LIMIT 10
        """))
        
        for r in result2:
            print(f"  {r[0]} | TF: {r[1]} ({r[2]}) | {r[3]} registros")

### 🔧 Diagnóstico: Verificar conexão da sessão ORM

In [ ]:
# Verificar qual banco a sessão está usando
from newapp.src.database.models import AssetsRates

# Tentar query direta pelo ORM
count = db.query(AssetsRates).count()
print(f"📊 Total de registros na tabela assets_rates (via ORM): {count}")

# Filtrar especificamente
count_wdo = db.query(AssetsRates).filter(
    AssetsRates.symbol == "WDO$",
    AssetsRates.timeframe == 5
).count()

print(f"📊 Registros WDO$ timeframe=5 (via ORM): {count_wdo}")

# Buscar uma amostra
sample = db.query(AssetsRates).filter(
    AssetsRates.symbol == "WDO$",
    AssetsRates.timeframe == 5
).limit(3).all()

if sample:
    print(f"\n✅ DADOS ENCONTRADOS via ORM!")
    print("=" * 70)
    for rec in sample:
        print(f"  {rec.symbol} | TF: {rec.timeframe} ({rec.timeframe_str}) | {rec.timestamp} | Close: {rec.close}")
else:
    print("\n❌ Nenhum dado encontrado via ORM")

In [ ]:
# Teste SQL RAW pelo engine criado
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM assets_rates"))
    total = result.fetchone()[0]
    print(f"📊 Total na tabela (SQL direto pelo engine): {total}")
    
    result = conn.execute(text("""
        SELECT COUNT(*) FROM assets_rates 
        WHERE symbol = 'WDO$' AND timeframe = 5
    """))
    count_wdo = result.fetchone()[0]
    print(f"📊 WDO$ timeframe=5 (SQL direto): {count_wdo}")

In [ ]:
# Verificar EXATAMENTE qual arquivo o engine está usando
print(f"🔍 Engine URL: {engine.url}")
print(f"🔍 Working directory: {Path.cwd()}")
print(f"🔍 DB_PATH configurado: {DB_PATH}")
print(f"🔍 DB_PATH absoluto: {DB_PATH.resolve()}")
print(f"🔍 DB_PATH size: {DB_PATH.stat().st_size / 1024 / 1024:.2f} MB")

### ⚠️ PROBLEMA ENCONTRADO!

O engine está usando caminho **relativo** (`./wtnps_trade.db`), que muda dependendo do diretório de execução.

**Solução:** Forçar caminho absoluto antes de criar a sessão.

## 2. Método: save_rates_dataframe()

Salva um DataFrame com dados OHLC no banco de dados.

**Parâmetros:**
- `db`: Sessão do banco
- `df`: DataFrame com colunas (time, open, high, low, close, tick_volume, volume, spread)
- `symbol`: Símbolo do ativo (ex: "WDO$")
- `timeframe`: Timeframe inteiro (ex: 5 para M5)

In [ ]:
# OPCIONAL: Criar DataFrame de exemplo (caso queira adicionar novos dados)
# Descomente as linhas abaixo para inserir dados de teste

# sample_data = pd.DataFrame({
#     'open': [5400.0, 5405.0, 5410.0, 5408.0, 5412.0],
#     'high': [5406.0, 5411.0, 5415.0, 5413.0, 5417.0],
#     'low': [5399.0, 5404.0, 5409.0, 5407.0, 5411.0],
#     'close': [5405.0, 5410.0, 5408.0, 5412.0, 5416.0],
#     'tick_volume': [1500, 1800, 1600, 1700, 2000],
#     'volume': [150, 180, 160, 170, 200],
#     'spread': [2, 2, 3, 2, 2],
# }, index=pd.date_range(start='2025-11-23 10:00', periods=5, freq='5min'))

# print("📊 DataFrame de exemplo criado:")
# print(sample_data)
# print(f"\n✅ Total de registros: {len(sample_data)}")

print("⏭️ Pulando criação de dados de exemplo")
print("📌 Usando dados REAIS já existentes no banco")

In [ ]:
# OPCIONAL: Salvar dados de exemplo no banco
# Descomente para salvar os dados criados acima

# count = AssetsRatesRepository.save_rates_dataframe(
#     db=db,
#     df=sample_data,
#     symbol="WDO$",
#     timeframe=5
# )

# print(f"✅ {count} registros salvos no banco de dados")
# print(f"📍 Symbol: WDO$")
# print(f"📍 Timeframe: 5 (M5)")

print("⏭️ Pulando inserção de dados de exemplo")
print("📌 Trabalhando com dados REAIS do banco")

## Método: get_all_rates()

Recupera todos os rates de um símbolo e timeframe específicos.

**Parâmetros:**
- `db`: Sessão do banco
- `symbol`: Símbolo do ativo
- `timeframe`: Timeframe inteiro

**Retorna:** DataFrame com todos os dados OHLC + indicadores

In [ ]:
# Primeiro, vamos verificar quais símbolos e timeframes existem no banco
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT symbol, timeframe, COUNT(*) as count 
        FROM assets_rates 
        GROUP BY symbol, timeframe
        ORDER BY count DESC
    """))
    
    print("📊 DADOS DISPONÍVEIS NO BANCO:")
    print("=" * 70)
    for row in result:
        print(f"  Symbol: {row[0]:<10} | Timeframe: {row[1]:<5} | Registros: {row[2]}")

print("\n" + "=" * 70)

# Buscar TODOS os rates do símbolo WDO$ com timeframe 5 (M5)
df_all_rates = AssetsRatesRepository.get_all_rates(
    db=db,
    symbol="WDO$",
    timeframe=5  # INTEGER, não string!
)

print(f"\n📊 Total de registros recuperados: {len(df_all_rates)}")

if not df_all_rates.empty:
    print(f"\n🔍 Primeiros 5 registros:")
    print(df_all_rates.head())
    print(f"\n🔍 Últimos 5 registros:")
    print(df_all_rates.tail())
    print(f"\n📋 Colunas disponíveis:")
    print(df_all_rates.columns.tolist())
else:
    print("⚠️ Nenhum registro encontrado! Verifique symbol e timeframe.")

## 4. Método: get_rates_range()

Recupera rates dentro de um intervalo de datas específico.

**Parâmetros:**
- `db`: Sessão do banco
- `symbol`: Símbolo do ativo
- `timeframe`: Timeframe inteiro
- `start_date`: Data inicial (datetime)
- `end_date`: Data final (datetime)

**Retorna:** DataFrame com dados OHLCV no intervalo especificado

In [ ]:
# Usar intervalo de datas REAL dos dados existentes
# Pegando as primeiras 10 datas do DataFrame

if not df_all_rates.empty:
    # Pegar intervalo real dos dados
    start_date = df_all_rates.index[0]
    end_date = df_all_rates.index[min(9, len(df_all_rates)-1)]  # Primeiras 10 barras
    
    # Buscar rates no intervalo
    df_range = AssetsRatesRepository.get_rates_range(
        db=db,
        symbol="WDO$",
        timeframe=5,
        start_date=start_date,
        end_date=end_date
    )
    
    print(f"📅 Intervalo: {start_date} até {end_date}")
    print(f"📊 Registros encontrados: {len(df_range)}")
    print(f"\n🔍 Dados:")
    print(df_range)
else:
    print("⚠️ Sem dados no DataFrame. Execute a célula anterior primeiro.")

## 5. Método: get_rates_indicators_range()

Recupera apenas indicadores técnicos (close, ema_9, sma_20, sma_50) dentro de um intervalo de datas.

**Parâmetros:**
- `db`: Sessão do banco
- `symbol`: Símbolo do ativo
- `timeframe`: Timeframe inteiro
- `start_date`: Data inicial (datetime)
- `end_date`: Data final (datetime)

**Retorna:** DataFrame com close e indicadores técnicos

In [ ]:
# Buscar indicadores técnicos no mesmo intervalo
if not df_all_rates.empty:
    df_indicators = AssetsRatesRepository.get_rates_indicators_range(
        db=db,
        symbol="WDO$",
        timeframe=5,
        start_date=start_date,
        end_date=end_date
    )
    
    print(f"📅 Intervalo: {start_date} até {end_date}")
    print(f"📊 Registros encontrados: {len(df_indicators)}")
    print(f"\n📈 Indicadores:")
    print(df_indicators)
    print(f"\n📋 Colunas: {df_indicators.columns.tolist()}")
    
    # Verificar se os indicadores estão preenchidos
    print(f"\n🔍 Valores não-nulos:")
    print(df_indicators.notna().sum())
else:
    print("⚠️ Sem dados no DataFrame. Execute a célula anterior primeiro.")

## 5.1. Atualizar Indicadores Técnicos com MarketContextAnalyzer

Este método calcula e atualiza os indicadores técnicos (EMA9, SMA20, SMA50) e níveis de suporte/resistência usando a classe `MarketContextAnalyzer`.

**O que faz:**
- Busca todos os rates do símbolo/timeframe
- Calcula indicadores usando `MarketContextAnalyzer`
- Atualiza os campos: `ema_9`, `sma_20`, `sma_50`, `support_level`, `resistance_level`
- Retorna número de registros atualizados

**Parâmetros:**
- `db`: Sessão do banco
- `symbol`: Símbolo do ativo
- `timeframe`: Timeframe inteiro
- `analyzer`: Instância opcional de MarketContextAnalyzer (cria padrão se None)

In [ ]:
# Importar MarketContextAnalyzer
from newapp.src.analysis.context_analyzer import MarketContextAnalyzer

# Verificar dados ANTES da atualização
print("📊 ANTES DA ATUALIZAÇÃO DE INDICADORES")
print("=" * 70)

if not df_all_rates.empty:
    # Mostrar primeiras 5 linhas com indicadores atuais
    cols_to_show = ['close', 'ema_9', 'sma_20', 'sma_50', 'sma_200', 'support_level', 'resistance_level']
    print(df_all_rates[cols_to_show].head())
    
    # Verificar quantos têm indicadores preenchidos
    print(f"\n🔍 Registros com EMA_9 não-nulo: {df_all_rates['ema_9'].notna().sum()}")
    print(f"🔍 Registros com SMA_20 não-nulo: {df_all_rates['sma_20'].notna().sum()}")
    print(f"🔍 Registros com SMA_50 não-nulo: {df_all_rates['sma_50'].notna().sum()}")
else:
    print("⚠️ Nenhum dado disponível. Execute as células anteriores primeiro.")

print("\n" + "=" * 70)
print("🚀 INICIANDO CÁLCULO DE INDICADORES...")
print("=" * 70)

In [ ]:


# Criar analyzer customizado (opcional - deixe None para usar padrões)
analyzer = MarketContextAnalyzer(
    ema_fast=9,      # EMA rápida
    sma_fast=20,     # SMA rápida
    sma_slow=50,     # SMA lenta
    sma_lookback=25, # Janela para inclinação da SMA
    rsi_period=14,   # Período do RSI (não salvo no banco, mas calculado internamente)
    lookback_levels=30  # Janela para suporte/resistência
)

# Calcular e atualizar indicadores no banco
count = AssetsRatesRepository.update_indicators_with_analyzer(
    db=db,
    symbol="WDO$",
    timeframe=5,
    analyzer=analyzer
)

print(f"\n✅ {count} registros atualizados com indicadores técnicos!")
print("=" * 70)

# BUSCAR DADOS ATUALIZADOS
print("\n🔄 Recarregando dados do banco...")
df_updated = AssetsRatesRepository.get_all_rates(db, "WDO$", 5)

print("\n📊 DEPOIS DA ATUALIZAÇÃO DE INDICADORES")
print("=" * 70)

if not df_updated.empty:
    # Mostrar primeiras e últimas 5 linhas
    print("\n🔝 PRIMEIRAS 5 BARRAS:")
    print(df_updated[cols_to_show].head())
    
    print("\n⬇️ ÚLTIMAS 5 BARRAS:")
    print(df_updated[cols_to_show].tail())
    
    # Estatísticas dos indicadores
    print(f"\n📈 ESTATÍSTICAS DOS INDICADORES:")
    print("=" * 70)
    print(f"EMA_9  - Min: {df_updated['ema_9'].min():.2f} | Max: {df_updated['ema_9'].max():.2f} | Mean: {df_updated['ema_9'].mean():.2f}")
    print(f"SMA_20 - Min: {df_updated['sma_20'].min():.2f} | Max: {df_updated['sma_20'].max():.2f} | Mean: {df_updated['sma_20'].mean():.2f}")
    print(f"SMA_50 - Min: {df_updated['sma_50'].min():.2f} | Max: {df_updated['sma_50'].max():.2f} | Mean: {df_updated['sma_50'].mean():.2f}")
    
    # Contar níveis de suporte/resistência
    support_count = df_updated['support_level'].sum()
    resistance_count = df_updated['resistance_level'].sum()
    print(f"\n🎯 Níveis de Suporte identificados: {support_count}")
    print(f"🎯 Níveis de Resistência identificados: {resistance_count}")
    
    # Verificar valores não-nulos
    print(f"\n✅ Registros com EMA_9 não-nulo: {df_updated['ema_9'].notna().sum()} / {len(df_updated)}")
    print(f"✅ Registros com SMA_20 não-nulo: {df_updated['sma_20'].notna().sum()} / {len(df_updated)}")
    print(f"✅ Registros com SMA_50 não-nulo: {df_updated['sma_50'].notna().sum()} / {len(df_updated)}")
else:
    print("⚠️ Erro ao recarregar dados")

print("\n" + "=" * 70)
print("✅ Atualização de indicadores concluída!")


## 6. Visualização dos Dados

Vamos visualizar graficamente os dados recuperados.

In [ ]:
import matplotlib.pyplot as plt

# Plotar preços OHLC
if not df_range.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df_range.index, df_range['close'], 'b-', linewidth=2, label='Close')
    ax.plot(df_range.index, df_range['high'], 'g--', alpha=0.5, label='High')
    ax.plot(df_range.index, df_range['low'], 'r--', alpha=0.5, label='Low')
    
    ax.set_title('Preços OHLC - WDO$ M5', fontsize=14, fontweight='bold')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Preço')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("✅ Gráfico gerado com sucesso")
else:
    print("⚠️ Sem dados para plotar")

## 7. Estatísticas dos Dados

In [ ]:
if not df_all_rates.empty:
    print("📊 ESTATÍSTICAS DESCRITIVAS")
    print("=" * 60)
    print(df_all_rates[['open', 'high', 'low', 'close', 'volume']].describe())
    
    print("\n📈 VARIAÇÃO DE PREÇOS")
    print("=" * 60)
    print(f"Preço mínimo: {df_all_rates['low'].min():.2f}")
    print(f"Preço máximo: {df_all_rates['high'].max():.2f}")
    print(f"Variação total: {df_all_rates['high'].max() - df_all_rates['low'].min():.2f}")
    print(f"Volume total: {df_all_rates['volume'].sum()}")
else:
    print("⚠️ Sem dados no banco")

## 8. Verificar Estrutura da Tabela

Vamos inspecionar a estrutura da tabela `assets_rates` diretamente no banco.

In [ ]:
from sqlalchemy import inspect, text

# Inspecionar tabela assets_rates
inspector = inspect(engine)

if 'assets_rates' in inspector.get_table_names():
    print("📋 ESTRUTURA DA TABELA: assets_rates")
    print("=" * 60)
    
    columns = inspector.get_columns('assets_rates')
    for col in columns:
        print(f"  • {col['name']:<20} {col['type']}")
    
    print("\n🔑 ÍNDICES:")
    indexes = inspector.get_indexes('assets_rates')
    for idx in indexes:
        print(f"  • {idx['name']}: {idx['column_names']}")
    
    # Contar registros
    with engine.connect() as conn:
        result = conn.execute(text("SELECT COUNT(*) FROM assets_rates"))
        count = result.fetchone()[0]
        print(f"\n📊 Total de registros na tabela: {count}")
else:
    print("⚠️ Tabela 'assets_rates' não encontrada no banco")

## 9. Limpeza e Fechamento

Fechar a sessão do banco de dados.

In [ ]:
# Fechar a sessão do banco
db.close()
print("✅ Sessão do banco de dados fechada")
print("✅ Notebook finalizado com sucesso")

---

## ⚠️ PONTOS IMPORTANTES

### 1. **Timeframe: Integer vs String**
A tabela `assets_rates` tem DOIS campos de timeframe:
- `timeframe` (Integer): **5** para M5, **15** para M15, etc.
- `timeframe_str` (String): **"M5"**, **"M15"**, etc.

**SEMPRE use o INTEGER nos métodos do repository!**
```python
# ✅ CORRETO
df = AssetsRatesRepository.get_all_rates(db, "WDO$", 5)

# ❌ ERRADO
df = AssetsRatesRepository.get_all_rates(db, "WDO$", "M5")
```

### 2. **Dados de Exemplo vs Dados Reais**
- As células agora usam os dados **REAIS** já existentes no banco
- Para adicionar dados de teste, descomente as células marcadas como OPCIONAL

### 3. **Verificação de Dados**
Execute a célula de "Verificação Rápida" logo após a conexão para ver:
- Quais tabelas existem
- Quantos registros cada uma tem
- Amostra dos dados

### 4. **Troubleshooting**
Se `get_all_rates()` retornar 0 registros:
1. Verifique o símbolo (case-sensitive: "WDO$" ≠ "wdo$")
2. Confirme o timeframe (INTEGER, não string)
3. Execute a consulta SQL direta para verificar:
```python
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT COUNT(*) FROM assets_rates 
        WHERE symbol = 'WDO$' AND timeframe = 5
    """))
    print(f"Registros: {result.fetchone()[0]}")
```